# Spot-Checking with Dimensionality Reduction

This notebook evaluates the impact of different **dimensionality reduction techniques** on classification performance, using a reusable pipeline built with `imblearn` and `scikit-learn`.

**Data flow:**
- Input: `kidney_disease_encoded.csv` : dataset with all features already encoded (generated by `Encode_Categorical_Values.ipynb`)
- Normalization is applied **inside the pipeline**, fitted only on the training data of each fold (no data leakage)
- **No balancing** is applied : identified as the best-performing strategy in `Spot_Checking_Balancing.ipynb`
- Dimensionality reduction is applied **after scaling**, fitted only on the training data of each fold

**Approach:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision and Recall
- DR technique and classifier can be swapped without changing the pipeline structure

**Dimensionality reduction techniques evaluated:**
- **PCA** : unsupervised linear transformation into principal components
- **Filter (f_classif)** : `SelectKBest` with ANOVA F-test; model-agnostic, ranks features by statistical relevance to the target
- **Filter (mutual_info)** : `SelectKBest` with Mutual Information; model-agnostic, captures non-linear dependencies
- **Embedded (RF)** : `SelectFromModel` with RandomForest; uses `feature_importances_` from a tree ensemble
- **Embedded (LinearSVC)** : `SelectFromModel` with LinearSVC; uses `coef_` magnitude from a linear model
- **Wrapper (RFECV)** : Recursive Feature Elimination with CV; automatically determines the optimal feature subset size

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    SelectFromModel, RFECV,
)

from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "reports" / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from src.modeling_utils import make_modeling_pipeline, make_stratified_cv
from src.visualization_utils import save_dataframe_as_figure

import warnings
warnings.filterwarnings('ignore')

## 2. Load and Prepare Data

In [ ]:
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, not ckd=1 (or vice versa)

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Pipeline

The `make_pipeline_with_dr` function is a notebook-local wrapper around the shared helper in `src/modeling_utils.py` and encapsulates:
1. **`MinMaxScaler`** : scaling fitted only on the training data of each fold
2. **`DR step`** : any sklearn-compatible transformer (PCA, `SelectKBest`, `SelectFromModel`, or `RFECV`)
3. **`Classifier`** : interchangeable without changing the rest of the pipeline

The shared utility keeps the step order aligned with the other modeling notebooks. No balancing is applied (`sampler=None`), consistent with the best strategy found in `Spot_Checking_Balancing.ipynb`.

In [ ]:
def make_pipeline_with_dr(classifier, reducer, sampler=None):
    return make_modeling_pipeline(
        classifier=classifier,
        sampler=sampler,
        reducer=reducer,
    )

## 4. Algorithms Definition

In [ ]:
models = {
    "KNN":   KNeighborsClassifier(),
    "DTree": DecisionTreeClassifier(random_state=42),
    "RF":    RandomForestClassifier(random_state=42),
    "SVM":   SVC(random_state=42),
    "NB":    GaussianNB(),
}

## 5. Dimensionality Reduction Techniques

Four families of DR techniques are evaluated:

| Family | Technique | Mechanism |
|---|---|---|
| Transformation | PCA | Linear projection into principal components |
| Filter | SelectKBest (f_classif) | ANOVA F-test (statistical relevance to target) |
| Filter | SelectKBest (mutual_info) | Mutual information (captures non-linear dependencies) |
| Embedded | SelectFromModel (RF) | `feature_importances_` from a RandomForest |
| Embedded | SelectFromModel (LinearSVC) | `coef_` magnitude from a linear model |
| Wrapper | RFECV | Recursive elimination; CV determines optimal subset size |

> **Note on RFECV:** performs its own internal cross-validation (cv=3) to find the optimal number of features. When nested inside the outer `cross_validate` (k=5), computation is heavier than the other techniques.

In [ ]:
n_features = X.shape[1]
# k for filter methods: half the original features (reasonable default for spot-checking)
k_half = max(1, n_features // 2)

dr_techniques = {
    # --- Transformation (PCA) ---
    "PCA (90% var)": PCA(n_components=0.90, random_state=42),
    "PCA (95% var)": PCA(n_components=0.95, random_state=42),

    # --- Filter methods ---
    "Filter (f_classif)":   SelectKBest(f_classif, k=k_half),
    "Filter (mutual_info)": SelectKBest(mutual_info_classif, k=k_half),

    # --- Embedded methods ---
    "Embedded (RF)":       SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42)),
    "Embedded (LinearSVC)": SelectFromModel(LinearSVC(random_state=42, max_iter=2000)),

    # --- Wrapper ---
    # cv=3 inside RFECV to limit computation cost (nested inside outer k=5 CV)
    "Wrapper (RFECV)": RFECV(
        estimator=RandomForestClassifier(n_estimators=50, random_state=42),
        cv=make_stratified_cv(n_splits=3),
        scoring="f1",
    ),
}

print(f"Original number of features: {n_features}")
print(f"k selected for filter methods (n // 2): {k_half}")
print(f"DR techniques defined: {list(dr_techniques.keys())}")

## 6. Spot-Checking with K-Fold Cross Validation

In [ ]:
cv = make_stratified_cv()
scoring = ['accuracy', 'precision', 'recall', 'f1']

# dr_results[dr_technique][model] = cross_validate scores dict
dr_results = {}

for dr_name, reducer in dr_techniques.items():
    dr_results[dr_name] = {}
    for model_name, model_candidate in models.items():
        pipeline = make_pipeline_with_dr(
            model_candidate,
            reducer=reducer,
            sampler=None,  # No balancing — best strategy from Spot_Checking_Balancing.ipynb
        )
        scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=False)
        dr_results[dr_name][model_name] = scores
    print(f"Done: {dr_name}")

print("\nAll done.")

## 7. Summary Table

In [ ]:
rows = []
for dr_name, model_results in dr_results.items():
    for model_name, scores in model_results.items():
        row = {"DR Technique": dr_name, "Model": model_name}
        for metric in scoring:
            row[f"{metric}_mean"] = np.mean(scores[f"test_{metric}"])
            row[f"{metric}_std"]  = np.std(scores[f"test_{metric}"])
        rows.append(row)

dr_summary_df = (
    pd.DataFrame(rows)
    .sort_values(["DR Technique", "f1_mean"], ascending=[True, False])
    .set_index(["DR Technique", "Model"])
)

display(dr_summary_df)

## 8. Boxplot (CV)

In [ ]:
n_techniques = len(dr_results)
ncols = 3
nrows = int(np.ceil(n_techniques / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for ax, (dr_name, model_results) in zip(axes, dr_results.items()):
    f1_data = {
        model_name: scores["test_f1"].tolist()
        for model_name, scores in model_results.items()
    }
    pd.DataFrame(f1_data).boxplot(ax=ax)
    ax.set_title(dr_name)
    ax.set_ylabel("F1-score")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=15)

# Hide unused subplots
for ax in axes[n_techniques:]:
    ax.set_visible(False)

plt.suptitle("F1-score Distribution per DR Technique (CV, No Balancing)", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "spot_checking_dimensionality_reduction_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Ranking

In [ ]:
# Ranking of DR techniques based on average F1-score across models, with tie-breaking by variability (std) of F1 between models.
# Main criterion: average F1-score of models for each technique
# Tie-breaking criterion: lower variability of F1 between models

summary_reset = dr_summary_df.reset_index()

dr_ranking_df = (
    summary_reset
    .groupby("DR Technique", as_index=False)
    .agg(
        accuracy_mean=("accuracy_mean", "mean"),
        precision_mean=("precision_mean", "mean"),
        recall_mean=("recall_mean", "mean"),
        f1_mean=("f1_mean", "mean"),          # average performance across models
        f1_std=("f1_mean", "std"),  # variability across models
    )
    .sort_values(["f1_mean", "f1_std"], ascending=[False, True])
    .reset_index(drop=True)
)

dr_ranking_df.index = dr_ranking_df.index + 1
dr_ranking_df.index.name = "Rank"

float_cols = dr_ranking_df.select_dtypes(include="float").columns.tolist()
ranking_style = dr_ranking_df.style.format("{:.4f}", subset=float_cols).background_gradient(cmap="RdYlGn", axis=0, subset=float_cols)
display(ranking_style)

save_dataframe_as_figure(
    dr_ranking_df,
    FIGURES_DIR / "spot_checking_dimensionality_reduction_ranking.png",
)